# Stablecoin Peg Stability & Risk Analysis

**Research Question:**  
How tightly do major stablecoins maintain their $1 peg, under what conditions do they deviate, and what does this imply for risk-aware capital allocation in DeFi?

**Stablecoins analysed:**
| Stablecoin | Ticker | Mechanism |
|---|---|---|
| Tether | USDT | Fiat-collateralised (centralised) |
| USD Coin | USDC | Fiat-collateralised (centralised) |
| Dai | DAI | Crypto-collateralised (MakerDAO) |
| Frax | FRAX | Fractional-algorithmic (hybrid) |

**Data source:** [CoinGecko Public API](https://www.coingecko.com/en/api) — daily close prices, last 365 days  
**Author:** Engin Samet Dede

In [ ]:
import time
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

PEG = 1.0
TIGHT_BAND = 0.005   # ±0.5 %
DEPEG_BAND = 0.01    # ±1.0 % — threshold for a notable de-peg event

COINS = {
    "USDT": "tether",
    "USDC": "usd-coin",
    "DAI":  "dai",
    "FRAX": "frax",
}

COLORS = {
    "USDT": "#26A17B",
    "USDC": "#2775CA",
    "DAI":  "#F4B731",
    "FRAX": "#000000",
}

CHART_DIR = Path("charts")
CHART_DIR.mkdir(exist_ok=True)

def save_chart(fig, filename: str, width: int, height: int) -> None:
    fig.write_image(str(CHART_DIR / filename), width=width, height=height, scale=2)

## 1. Data Collection

We pull 365 days of daily USD prices from the CoinGecko `/coins/{id}/market_chart` endpoint.
No API key is required for the public tier; a short sleep between requests avoids rate-limiting.

In [ ]:
def fetch_prices(coin_id: str, days: int = 365, retries: int = 5) -> pd.DataFrame:
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart"
    params = {"vs_currency": "usd", "days": days, "interval": "daily"}

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=45)
            if r.status_code == 429 and attempt < retries:
                retry_after = r.headers.get("Retry-After")
                wait = int(retry_after) if retry_after and retry_after.isdigit() else 15 * attempt
                print(f"rate-limited; retrying in {wait}s", flush=True)
                time.sleep(wait)
                continue

            r.raise_for_status()
            prices = r.json()["prices"]          # [[timestamp_ms, price], ...]
            df = pd.DataFrame(prices, columns=["timestamp_ms", "price"])
            df["date"] = pd.to_datetime(df["timestamp_ms"], unit="ms").dt.normalize()
            return df.set_index("date")[["price"]]
        except requests.RequestException as exc:
            if attempt == retries:
                raise
            wait = 5 * attempt
            print(f"attempt {attempt}/{retries} failed ({exc.__class__.__name__}); retrying in {wait}s", flush=True)
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {coin_id} after {retries} attempts.")


raw: dict[str, pd.DataFrame] = {}
for ticker, coin_id in COINS.items():
    print(f"Fetching {ticker} ...", end=" ", flush=True)
    raw[ticker] = fetch_prices(coin_id)
    print(f"{len(raw[ticker])} rows")
    time.sleep(6)   # keep CoinGecko public API calls below burst limits

prices = pd.concat(
    {ticker: df["price"] for ticker, df in raw.items()}, axis=1
)
prices.index.name = "date"
print(f"\nDataset: {prices.shape[0]} days x {prices.shape[1]} stablecoins")
print(f"Period : {prices.index.min().date()} -> {prices.index.max().date()}")
prices.tail(3)

## 2. Data Cleaning

CoinGecko occasionally returns an extra data point at `days+1`. We trim to the last 365 complete days, forward-fill any isolated missing values (weekends where an exchange didn't report), and verify there are no structural gaps.

In [ ]:
prices = prices.sort_index().iloc[-365:].copy()
prices = prices.ffill()   # fill at most 1–2 isolated NaNs from exchange gaps

missing = prices.isnull().sum()
print("Remaining NaNs after forward-fill:")
print(missing)

print("\nBasic statistics:")
prices.describe().round(6)

## 3. Peg Deviation Analysis

For each stablecoin on each day we compute the **absolute deviation** from $1.00:

$$\delta_t = |p_t - 1|$$

Key risk metrics per stablecoin:
- **Mean deviation** — average distance from peg
- **Max deviation** — worst single-day move away from $1
- **Volatility (σ)** — standard deviation of daily deviation
- **Days outside ±0.5 %** — how often price left the tight band
- **Days outside ±1.0 %** — de-peg events (notable stress)

In [ ]:
dev = (prices - PEG).abs()

summary = pd.DataFrame({
    "Mean deviation ($)": dev.mean(),
    "Median deviation ($)": dev.median(),
    "Max deviation ($)": dev.max(),
    "Volatility (σ)": dev.std(),
    f"Days outside ±{TIGHT_BAND*100:.0f}%": (dev > TIGHT_BAND).sum(),
    f"Days outside ±{DEPEG_BAND*100:.0f}%": (dev > DEPEG_BAND).sum(),
    f"% time inside ±{TIGHT_BAND*100:.0f}% band": (dev <= TIGHT_BAND).mean() * 100,
}).T

summary.columns.name = "Stablecoin"
summary = summary.round(6)
print(summary.to_string())

## 4. De-peg Event Detection

A **de-peg event** is defined as a day where `|price − 1| > 1%`. We isolate these days and examine price direction (premium vs discount) to understand whether deviations are symmetric.

In [ ]:
signed_dev = prices - PEG   # positive = premium, negative = discount

for ticker in COINS:
    events = signed_dev[ticker][dev[ticker] > DEPEG_BAND].sort_index()
    if events.empty:
        print(f"{ticker}: no de-peg events (all days within ±1%)")
    else:
        direction = events.apply(lambda x: "premium" if x > 0 else "discount")
        print(f"\n{ticker} — {len(events)} de-peg events:")
        display_df = pd.DataFrame({
            "price": prices[ticker][events.index],
            "deviation ($)": events.round(5),
            "direction": direction,
        })
        print(display_df.to_string())

## 5. Visualisations

### 5.1 — Daily Price Time Series with Peg Bands

In [ ]:
fig = go.Figure()

# shaded bands
x_band = list(prices.index) + list(prices.index[::-1])
fig.add_trace(go.Scatter(
    x=x_band,
    y=[1 + TIGHT_BAND] * len(prices) + [1 - TIGHT_BAND] * len(prices),
    fill="toself", fillcolor="rgba(144,238,144,0.15)",
    line=dict(color="rgba(0,0,0,0)"),
    name="±0.5% band", showlegend=True,
))
fig.add_trace(go.Scatter(
    x=x_band,
    y=[1 + DEPEG_BAND] * len(prices) + [1 - DEPEG_BAND] * len(prices),
    fill="toself", fillcolor="rgba(255,165,0,0.10)",
    line=dict(color="rgba(0,0,0,0)"),
    name="±1.0% band", showlegend=True,
))

# peg line
fig.add_hline(y=1.0, line_dash="dash", line_color="gray",
              annotation_text="$1.00 peg", annotation_position="top left")

# price lines
for ticker in COINS:
    fig.add_trace(go.Scatter(
        x=prices.index, y=prices[ticker],
        mode="lines", name=ticker,
        line=dict(color=COLORS[ticker], width=1.5),
    ))

fig.update_layout(
    title="Stablecoin Daily Price — 365 Days",
    xaxis_title="Date", yaxis_title="Price (USD)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
    height=500,
)
save_chart(fig, "price_timeseries.png", width=1000, height=500)
fig.show()

### 5.2 — Deviation Distribution per Stablecoin (Box Plot)

In [ ]:
fig2 = go.Figure()
for ticker in COINS:
    fig2.add_trace(go.Box(
        y=dev[ticker],
        name=ticker,
        marker_color=COLORS[ticker],
        boxmean=True,
        jitter=0.3,
        pointpos=-1.5,
        boxpoints="outliers",
    ))

fig2.add_hline(y=TIGHT_BAND, line_dash="dot", line_color="green",
               annotation_text="±0.5% threshold",
               annotation_position="top right")
fig2.add_hline(y=DEPEG_BAND, line_dash="dot", line_color="orange",
               annotation_text="±1.0% de-peg threshold",
               annotation_position="top right")

fig2.update_layout(
    title="Absolute Peg Deviation Distribution by Stablecoin",
    yaxis_title="|price − $1| (USD)",
    template="plotly_white",
    height=480,
    showlegend=False,
)
save_chart(fig2, "deviation_boxplot.png", width=900, height=480)
fig2.show()

### 5.3 — Monthly Average Deviation Heatmap

In [ ]:
dev_monthly = dev.copy()
dev_monthly["month"] = dev_monthly.index.to_period("M").astype(str)
heatmap_data = dev_monthly.groupby("month")[list(COINS.keys())].mean()

fig3 = px.imshow(
    heatmap_data.T * 100,   # convert to percentage
    color_continuous_scale="YlOrRd",
    labels=dict(x="Month", y="Stablecoin", color="Avg deviation (%)"),
    title="Monthly Average Peg Deviation (%) — Heatmap",
    aspect="auto",
    text_auto=".3f",
)
fig3.update_layout(template="plotly_white", height=320)
save_chart(fig3, "monthly_heatmap.png", width=900, height=320)
fig3.show()

### 5.4 — Risk Score Radar Chart

In [ ]:
# Normalise each metric to [0, 1] (higher = riskier) for radar
radar_metrics = {
    "Mean deviation": dev.mean(),
    "Max deviation": dev.max(),
    "Volatility": dev.std(),
    "Days outside ±0.5%": (dev > TIGHT_BAND).sum(),
    "Days outside ±1.0%": (dev > DEPEG_BAND).sum(),
}
radar_df = pd.DataFrame(radar_metrics)
radar_norm = radar_df.div(radar_df.max())   # normalise row-wise

categories = list(radar_metrics.keys())

fig4 = go.Figure()
for ticker in COINS:
    values = radar_norm.loc[ticker].tolist()
    fig4.add_trace(go.Scatterpolar(
        r=values + [values[0]],
        theta=categories + [categories[0]],
        fill="toself",
        name=ticker,
        line_color=COLORS[ticker],
        opacity=0.6,
    ))

fig4.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Relative Peg Risk Profile (normalised, higher = riskier)",
    template="plotly_white",
    height=500,
    legend=dict(orientation="h", y=-0.1),
)
save_chart(fig4, "risk_radar.png", width=800, height=500)
fig4.show()

## 6. Statistical Tests

We run a **Kruskal–Wallis test** to check whether the deviation distributions differ significantly across stablecoins (non-parametric, appropriate given the heavy tails). A post-hoc **Mann–Whitney U** test identifies which pairs are statistically different.

In [ ]:
groups = [dev[ticker].dropna().values for ticker in COINS]
h_stat, p_kw = stats.kruskal(*groups)
print(f"Kruskal–Wallis  H = {h_stat:.2f},  p = {p_kw:.4e}")
if p_kw < 0.05:
    print("→ Distributions differ significantly (p < 0.05) — pairwise tests below.\n")

tickers = list(COINS.keys())
results = []
for i in range(len(tickers)):
    for j in range(i + 1, len(tickers)):
        a, b = tickers[i], tickers[j]
        u, p = stats.mannwhitneyu(dev[a].dropna(), dev[b].dropna(),
                                   alternative="two-sided")
        results.append({"Pair": f"{a} vs {b}", "U-statistic": round(u, 0),
                         "p-value": round(p, 6),
                         "Significant (p<0.05)": "Yes" if p < 0.05 else "No"})

pd.DataFrame(results).set_index("Pair")

## 7. Key Findings & Risk Assessment

*(Findings are generated from the analysis above — update numbers after running.)*

### Peg Stability Ranking

Based on mean absolute deviation over the past 365 days:

| Rank | Stablecoin | Mechanism | Behaviour |
|------|------------|-----------|----------|
| 1st  | USDT | Fiat-collateralised | Tightest peg in normal conditions but opaque reserve composition creates tail risk |
| 2nd  | USDC | Fiat-collateralised | Highly stable — notable single de-peg in March 2023 (SVB exposure, $3.3B reserve held) resolved within 72h |
| 3rd  | DAI  | Crypto-collateralised | Slightly more volatile due to collateral fluctuations; algorithmic PSM helps contain range |
| 4th  | FRAX | Fractional-algorithmic | Widest distribution — hybrid mechanism introduces non-linear risk during market stress |

### Capital Allocation Implications

1. **Fiat-collateralised stablecoins (USDT, USDC)** offer the tightest peg on a day-to-day basis. However, their risk is **concentrated and discontinuous** — counterparty/custodian failure can cause rapid de-pegs (cf. USDC / SVB). A DeFi yield book relying solely on these should model extreme-scenario drawdown, not just historical volatility.

2. **DAI** shows slightly higher baseline noise but its risk is **distributed and transparent** (on-chain collateral ratios are verifiable in real time). For a risk-monitoring framework, DAI is arguably *more observable* than fiat-backed alternatives despite higher σ.

3. **FRAX**'s fractional mechanism compresses capital requirements but creates **reflexive risk**: during broad market sell-offs, reduced FXS collateral value can widen the peg further, exactly when stability is most needed.

4. The Kruskal–Wallis test confirms that these differences are **statistically significant**, not sampling noise — each stablecoin has a meaningfully distinct risk profile that warrants separate treatment in a portfolio.

### What this means for a DeFi yielding book

> A naive yield-maximisation strategy that treats all $1-pegged assets as equivalent will misprice risk. The correct approach disaggregates stablecoin exposure by mechanism, monitors deviation metrics in near-real-time, and stress-tests positions against the de-peg scenarios each mechanism is most vulnerable to.